# 02 — Chains & Memory

**Goal**: Build conversational applications that remember context.

LLMs are stateless — they don't remember past interactions.
Memory components give them "short-term memory."

In [ ]:
from langchain_community.llms import Ollama
from langchain.prompts import PromptTemplate
from langchain.chains import ConversationChain
from langchain.memory import ConversationBufferMemory, ConversationSummaryMemory

llm = Ollama(model="llama3.2")
print("Ready")

## 1. The Problem: Stateless LLM

Each call is independent — no memory of previous turns.

In [ ]:
# First question
r1 = llm.invoke("My name is Alice. What is my name?")
print(f"Q1: {r1}")

# Second question — no context
r2 = llm.invoke("What is my name?")
print(f"Q2: {r2}")

## 2. Conversation Buffer Memory

Stores the entire conversation history and injects it into each prompt.

In [ ]:
memory = ConversationBufferMemory()
conversation = ConversationChain(llm=llm, memory=memory)

r1 = conversation.predict(input="Hi, my name is Alice and I love machine learning")
print(f"You: Hi, my name is Alice...")
print(f"Bot: {r1}\n")

r2 = conversation.predict(input="What is my name and what do I love?")
print(f"You: What is my name and what do I love?")
print(f"Bot: {r2}")

In [ ]:
# See what memory looks like internally
print("Current memory buffer:")
print(memory.buffer)

## 3. Conversation Summary Memory

Buffer memory grows forever (uses more tokens). Summary memory compresses history.

In [ ]:
summary_memory = ConversationSummaryMemory(llm=llm)
summary_convo = ConversationChain(llm=llm, memory=summary_memory)

summary_convo.predict(input="I'm Bob and I work as a data scientist")
summary_convo.predict(input="My favorite project is building recommendation systems")
summary_convo.predict(input="What do I do for work?")

print("Summary memory (compressed):")
print(summary_memory.buffer)

## 4. Conversation Buffer Window Memory

Only remembers the last K exchanges — limits token usage.

In [ ]:
from langchain.memory import ConversationBufferWindowMemory

window_memory = ConversationBufferWindowMemory(k=2)  # remember last 2 exchanges
window_convo = ConversationChain(llm=llm, memory=window_memory)

window_convo.predict(input="My name is Charlie")
window_convo.predict(input="I like pizza")
window_convo.predict(input="What is my favorite food?")
window_convo.predict(input="What is my name?")  # This might be forgotten!

## 5. Custom Memory-Powered Chain

Let's build a simple study assistant with memory.

In [ ]:
tutor_memory = ConversationBufferMemory()
tutor = ConversationChain(
    llm=llm,
    memory=tutor_memory,
    prompt=PromptTemplate(
        input_variables=["history", "input"],
        template="""You are a patient tutor. Use conversation history to help the student learn.

History:
{history}

Student: {input}
Tutor:"""
    )
)

print(tutor.predict(input="I want to learn about transformers"))
print("\n--- Next question ---\n")
print(tutor.predict(input="What is self-attention?"))

## 6. When to Use Which Memory

| Memory Type | Use Case | Cost |
|---|---|---|
| Buffer | Short chats, need full context | High (grows forever) |
| Buffer Window | Long conversations, recent context | Medium (fixed size) |
| Summary | Very long conversations | Low (compressed) |
| Vector Store Memory | RAG-like memory retrieval | Medium |
| Entity Memory | Remember specific facts about entities | Medium |

## Key Takeaways

1. **LLMs are stateless** — you must provide context each time
2. **ConversationBufferMemory** stores full history (easy but expensive)
3. **SummaryMemory** compresses history (good for long convos)
4. **WindowMemory** keeps only last K exchanges (good for short-term)
5. Memory is injected into the prompt as additional context

**Next**: Agents — let the LLM decide what tools to use →